In [4]:
# status_percentages.py
import pandas as pd
import os

def status_distribution(input_csv="script_5/transcript_clean_without_no_answer.csv"):
    """
    Выводит распределение статусов (в штуках и процентах) относительно ВСЕЙ выборки.
    """
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Файл не найден: {input_csv}")

    df = pd.read_csv(input_csv)

    if "status" not in df.columns:
        raise ValueError(f"Колонка 'status' отсутствует. Доступные: {list(df.columns)}")

    total = len(df)
    print(f"📁 Анализ файла: {input_csv}")
    print(f"📌 Общий объём выборки: {total} записей\n")

    # Считаем частоты (включая NaN как отдельную категорию)
    counts = df["status"].value_counts(dropna=False)
    percentages = (counts / total * 100).round(2)

    print("📊 Распределение статусов (меток Оли):")
    print("=" * 50)
    for status, count in counts.items():
        pct = percentages[status]
        status_str = str(status) if pd.notna(status) else "[НЕТ СТАТУСА]"
        print(f"{status_str:<30} : {int(count):>5} ({pct:>5}%)")
    print("=" * 50)
    print(f"Итого: {int(counts.sum())} записей")

    return counts, percentages

if __name__ == "__main__":
    status_distribution()

📁 Анализ файла: script_5/transcript_clean_without_no_answer.csv
📌 Общий объём выборки: 3204 записей

📊 Распределение статусов (меток Оли):
угроза оттока не определена    :  1520 (47.44%)
угроза оттока подтверждена     :   985 (30.74%)
угроза оттока не подтверждена  :   447 (13.95%)
угроза оттока не определена, требуется уточнение персонального менеджера :   231 ( 7.21%)
обновить контактные данные     :    18 ( 0.56%)
угроза оттока не подтверждена, только для работы оборудования :     2 ( 0.06%)
недозвон                       :     1 ( 0.03%)
Итого: 3204 записей


In [5]:
# script_stratified_audit.py
import pandas as pd
import os
from collections import OrderedDict

def format_dialogue(transcript):
    if pd.isna(transcript):
        return "[ПУСТО]"
    lines = []
    for part in transcript.split(";"):
        part = part.strip()
        if not part:
            continue
        if part.startswith(("bot:", "robot:")):
            text = part.split(":", 1)[1].strip()
            lines.append(f"[BOT]   {text}")
        elif part.startswith("human:"):
            text = part[6:].strip()
            lines.append(f"[HUMAN] {text}")
        else:
            lines.append(f"[???]   {part}")
    return "\n".join(lines) if lines else "[ДИАЛОГ ОТСУТСТВУЕТ]"

def create_stratified_audit(
    input_csv="script_5/transcript_clean_without_no_answer.csv",
    output_dir="script_stratified_audit",
    total_sample=320,
    random_seed=42,
    verbose=True
):
    df = pd.read_csv(input_csv)
    if "status" not in df.columns:
        raise ValueError("Колонка 'status' не найдена")

    # Подсчитываем исходные пропорции
    total = len(df)
    status_counts = df["status"].value_counts()
    status_proportions = (status_counts / total * total_sample).round().astype(int)

    # Корректируем, чтобы сумма = total_sample
    diff = total_sample - status_proportions.sum()
    if diff != 0:
        # Добавляем/убираем из самой большой категории
        largest = status_proportions.idxmax()
        status_proportions[largest] += diff

    # Убираем категории, где 0 (если появились)
    status_proportions = status_proportions[status_proportions > 0]

    if verbose:
        print(f"📁 Исходный файл: {input_csv}")
        print(f"📊 Целевая выборка: {total_sample} записей")
        print("Распределение по статусам:")
        for status, n in status_proportions.items():
            pct = n / total_sample * 100
            print(f"  '{status}' → {n} ({pct:.1f}%)")

    os.makedirs(output_dir, exist_ok=True)

    # Отбираем стратифицированную выборку
    sampled_dfs = []
    for status, n in status_proportions.items():
        group = df[df["status"] == status]
        if len(group) <= n:
            # Если в группе меньше, чем нужно — берём всё
            sample = group.copy()
        else:
            sample = group.sample(n=n, random_state=random_seed)
        sampled_dfs.append(sample)

    df_sample = pd.concat(sampled_dfs).reset_index(drop=True)

    # Сохраняем данные
    df_sample.to_csv(os.path.join(output_dir, "stratified_sample_320.csv"), index=False, encoding="utf-8")
    with open(os.path.join(output_dir, "ids_stratified.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(df_sample["id"].astype(str).tolist()))

    # Генерируем ЛОГ ПО СТАТУСАМ
    log_entries = []
    for status in status_proportions.index:
        group = df_sample[df_sample["status"] == status]
        log_entries.append(f"\n{'='*80}")
        log_entries.append(f"СТАТУС: {status} (n={len(group)})")
        log_entries.append('='*80 + "\n")
        
        for idx, row in group.iterrows():
            dialogue = format_dialogue(row["transcript"])
            log_entries.append(
                f"--- ID: {row['id']} ---\n"
                f"Диалог:\n{dialogue}\n"
                f"{'-'*60}\n"
            )

    log_path = os.path.join(output_dir, "stratified_audit_log.txt")
    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(log_entries))

    if verbose:
        print(f"\n✅ Выборка сохранена:")
        print(f"📄 Лог по статусам: {log_path}")
        print(f"📄 CSV: {output_dir}/stratified_sample_320.csv")

    return df_sample

if __name__ == "__main__":
    create_stratified_audit()

📁 Исходный файл: script_5/transcript_clean_without_no_answer.csv
📊 Целевая выборка: 320 записей
Распределение по статусам:
  'угроза оттока не определена' → 152 (47.5%)
  'угроза оттока подтверждена' → 98 (30.6%)
  'угроза оттока не подтверждена' → 45 (14.1%)
  'угроза оттока не определена, требуется уточнение персонального менеджера' → 23 (7.2%)
  'обновить контактные данные' → 2 (0.6%)

✅ Выборка сохранена:
📄 Лог по статусам: script_stratified_audit\stratified_audit_log.txt
📄 CSV: script_stratified_audit/stratified_sample_320.csv
